In [ ]:
## IMPORT BASE MODULES
import pandas as pd

import sys
import tomllib
from pathlib import Path

import openai

In [ ]:
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

print(root_dir)

In [ ]:
# Pfad zur config.toml
config_path = Path.cwd().parent / "config.toml"

with open(config_path, "rb") as f:
    config = tomllib.load(f)

# config ist jetzt ein normales Python-Dictionary
print(config)

In [ ]:
from src import (
    semgrep_analyzer,
    data_preprocessing,
    llm_analyzer,
    code_directory_analyzer
)

# **Aufbau des Experiments**
1. Semgrep-Run auf Sample Dateien
2. CodeCompletion auf Sample Dateien
3. Semgrep Run auf 2.
4. Evaluation + Visualisierung der Ergebnisse wird in Notebook `04_evaluation.ipynb` durchgeführt

# 1. Semgrep-Run auf Sample Dateien

Quelle: https://www.geeksforgeeks.org/python/difference-between-eval-and-ast-literal-eval-in-python/

Attribute-Beschreibungen für SemGrep: 
- https://docs.semgrep.dev/contributing/contributing-to-semgrep-rules-repository

Import Daten

In [ ]:
df_sample = pd.read_parquet(
    "../data/parquet/df_sample.parquet"
)

Konfigurationen für Semgrep Funktion

In [ ]:
all_paths = df_sample["id_file"].to_list()

home_dir = "../data/processed/"

command = config["semgrep"]["command"]

Durchführung der Semgrep Analyse

In [ ]:
semgrep_baseline = (
    semgrep_analyzer
    .semgrep_results(
        paths=all_paths, 
        # home_dir=home_dir, 
        semgrep_command=command
    )
)

Ergebnis in ein Dataframe umwandeln

In [ ]:
df_semgrep_baseline = (
    semgrep_analyzer
    .semgrep_results_to_dataframe(semgrep_baseline)
)

Semgrep Ergebnis mit weiteren Metadaten erweitern

In [ ]:
df_semgrep_baseline = (
    data_preprocessing
    .left_join(
        df_semgrep_baseline,
        df_sample,
        cols=["path", "id_file"]
    )
)

In [ ]:
df_semgrep_baseline

Aufbereitung Semgrep Ergebnis: Spalten mit `Listeneinträgen` werden in seperate Spalten eingefügt

In [ ]:
list_cols = ['cwe', 'owasp', 'technology', 'vulnerability_class', 'references']

df_semgrep_baseline = (
    data_preprocessing
    .expand_list_entries_to_different_columns(
        df_semgrep_baseline,
        list_cols,
    )
)

In [ ]:
df_semgrep_baseline

Zwischenspeichern des Ergebnisses

In [ ]:
df_semgrep_baseline.to_parquet("../data/parquet/df_semgrep_baseline.parquet")

# 2. CodeCompletion auf Sample Dateien
* Quelle: https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create 

Konfigurationen für die LLM

In [ ]:
key = config["llm"]["key"]

model = config["llm"]["model"]

client = openai.OpenAI(
    api_key=key,
    base_url="https://api.hrz.uni-giessen.de"
)

Completed Source Code soll separat in ein Verzeichnis gespeichert werden

In [ ]:
output_dir = Path("../data/CodeCompletion") 
output_dir.mkdir(parents=True, exist_ok=True)

Sequentielle Abarbeitung der Code Completion 

In [ ]:
i = 0
## Sequentielle Abarbeitung des Codes
for idx, row in df_sample.iterrows():
    input_path = Path(row["id_file"])
    file_name = row["document_id"]
    medium = row["medium"]
    level = row["level"]
    language = row["programming_language"]


    # Daten lesen
    try:
        with open(input_path, "r", encoding="utf-8", errors="replace") as f:
            file_content = f.read()
    except Exception as e:
        print(f"[{idx}] wird erstmal übersprungen wegen Lesefehler ({input_path}): {e}")
        continue

    # Code Completion
    try:
        result = (
            llm_analyzer
            .complete_code(
                client=client,
                model=model,
                raw_code=file_content,
                language=language,
                medium=medium,
                level=level
            )
        )
    
    except Exception as e:
        print(f"[{idx}] Fehler bei {input_path}: {e}")
        continue


    # Speichern mit id-Name 
    output_file = output_dir / f"{file_name}"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(result)
    i += 1

    print(f"[{i}] Erfolgreich verarbeitet: {output_file.name}")

# 3. Semgrep Run auf CodeCompletion

In [ ]:
code_completion_dir = Path("../data/CodeCompletion/")

# Alle Dateipfade rekursiv ermitteln
paths = [
    str(p.resolve())
    for p in code_completion_dir.rglob("*") 
    if p.is_file()
]


In [ ]:
home_dir = "../data/CodeCompletion/"

command = config["semgrep"]["command"]

semgrep_code_completion = (
    semgrep_analyzer
    .semgrep_results(
        paths=paths, 
        # home_dir=home_dir, 
        semgrep_command=command
    )
)

In [ ]:
df_semgrep_code_completion = (
    semgrep_analyzer
    .semgrep_results_to_dataframe(semgrep_code_completion)
)

In [ ]:
df_semgrep_code_completion

Erzeugung der ID

In [ ]:
df_semgrep_code_completion["document_id"] = (
    df_semgrep_code_completion["path"]
    .apply(lambda p: Path(p).name)
)
df_semgrep_code_completion

### Anwendung von `CodeDirectoryAnalyzer()`

* Begründung:
    - Die Erweiterung der Spalten auf Basis des Samples kann mit einem einfach `Join` nicht erfolgen
    - Code Completion verändert die Metadaten wie LOC oder Anzahl Characters -> müssen neu ermittelt werden

In [ ]:
analyzer = code_directory_analyzer.CodeDirectoryAnalyzer()
analyzer

Zu durchlaufende Pfade, um Metdadaten zu ermitteln

In [ ]:
paths = df_semgrep_code_completion.path.to_list()

Beginn mit der Analyse

In [ ]:
df_code_completion = (
    pd.DataFrame(
        analyzer
        .analyze_directory(
            path_of_data = "../data/CodeCompletion/"
        )
    )
)

In [ ]:
# # Zu entfernende Spalte
# drop_col = ["name_source"]

# # Namesnänderung
# rename_cols = {
#     "id_file": "absolute_path",
# }

# df_code_completion = (
#     df_code_completion
#     .drop(columns=drop_col)
#     .rename(columns=rename_cols)
# )

In [ ]:
df_code_completion

In [ ]:
df_code_completion.to_parquet("../data/parquet/df_code_completion_sample.parquet")

In [ ]:
df_semgrep_code_completion = (
    data_preprocessing
    .left_join(
        df_semgrep_code_completion,
        df_code_completion,
        cols=["document_id", "name_file"]
    )
)

In [ ]:
df_semgrep_code_completion

Aufbereitung der Ergebnisse von `df_semgrep_code_completion`
* Einige Spalten enthalten `Listeneinträge` die in separate Spalten eingefügt werden - Erleichterung bei Visualisierungen

In [ ]:
list_cols = ['cwe', 'owasp', 'technology', 'vulnerability_class', 'references']

df_semgrep_code_completion = (
    data_preprocessing
    .expand_list_entries_to_different_columns(
        df_semgrep_code_completion,
        list_cols,
    )
)

In [ ]:
df_semgrep_code_completion

In [ ]:
df_semgrep_baseline.to_parquet("../data/parquet/df_semgrep_baseline.parquet")

df_semgrep_code_completion.to_parquet("../data/parquet/df_semgrep_code_completion.parquet")